# unify_pums.ipynb

This notebook reads in the household and person-level PUMS for a given year, merges them, cleans up the ORIGIN/CHOSEN fields, and injects fields that will be used later on during modeling.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("../.."))


In [ ]:
sample_size = 1  # proportion of people in PUMS to consider
year = 2018
path = "us/pums_2018_raw.csv"

In [ ]:
def filter_cols(col):
    return "PERWTP" not in col


df = pd.read_csv(path, usecols=filter_cols)

In [7]:
df["PID"] = df["CBSERIAL"] * 1_000 + df["PERNUM"]
assert df["PID"].is_unique
df = df.set_index("PID")

In [8]:
df["MIGPLAC1"].value_counts()

MIGPLAC1
0      2788765
6        46187
48       36447
12       27232
36       20896
        ...   
330         75
350         73
520         71
623         66
622         61
Name: count, Length: 108, dtype: int64

In [9]:
df["PUMA"]

PID
2018010000049001    1600
2018010000058001    1900
2018010000219001    2000
2018010000246001    2400
2018010000251001    2701
                    ... 
2018001400326004     400
2018001400326005     400
2018001400502001     100
2018001400502002     100
2018001400515001     500
Name: PUMA, Length: 3214539, dtype: int64

In [ ]:
# origin is the MIGSP + MIGPUMA
# need ints since it is treated as a float by default
df["ORIGIN"] = df["MIGPLAC1"].astype(int).astype(str).str.zfill(2) + df[
    "MIGPUMA1"
].astype(int).astype(str).str.zfill(5)  # migpuma geography

# chosen is the current location, ST + PUMA
df["CHOSEN"] = df["STATEFIP"].astype(int).astype(str).str.zfill(2) + df["PUMA"].astype(
    int
).astype(str).str.zfill(5)  # puma geography

df["CHOSEN_MIGPUMA"] = df["STATEFIP"].astype(int).astype(str).str.zfill(2) + df[
    "MIGPUMANOW"
].astype(int).astype(str).str.zfill(5)

In [11]:
df["ORIGIN"].value_counts()

ORIGIN
0000000     2788765
0603700       10810
0400100        5848
1703400        5806
2500390        5542
             ...   
33000001         75
35000001         73
52000001         71
62300001         66
62200001         61
Name: count, Length: 1039, dtype: int64

In [ ]:
# backfill the stay origins to the MIGPUMA where they are currently (chosen == origin)
df["ORIGIN"] = np.where(
    df["ORIGIN"] == "0000000",
    df["CHOSEN_MIGPUMA"],
    df["ORIGIN"],
)
# fill in the origin state with this backfill in place
df["ORIGIN_STATE"] = np.where(
    df["ORIGIN"].str.len() == 7, df["ORIGIN"].str[:2].astype(int), -1
)

# define STAY as moving outside the MIGPUMA
df["STAY"] = df["CHOSEN_MIGPUMA"] == df["ORIGIN"]

In [13]:
df["ORIGIN"].value_counts()

ORIGIN
0603700     102203
2500390      49638
1703400      41902
0400100      41174
4804600      36741
             ...  
33000001        75
35000001        73
52000001        71
62300001        66
62200001        61
Name: count, Length: 1038, dtype: int64

In [14]:
df["ORIGIN_STATE"].value_counts()

ORIGIN_STATE
 6     377274
 48    265623
 12    199113
 36    197113
 42    128757
 17    126968
 39    118907
 37    101294
 13     99777
 26     99181
 34     88714
 51     84057
 53     75492
 25     69599
 4      68702
 18     67427
 47     67323
 29     62272
 24     59648
 55     59634
 27     55797
 8      55563
 45     49125
 1      47512
 21     45229
 22     43570
 41     41714
 40     37582
 9      36349
 19     32195
 49     31177
 5      30415
 20     29519
 28     29075
 32     28499
 31     19450
 35     19155
 54     18106
-1      17841
 16     16517
 15     14390
 33     13648
 23     13117
 44     10325
 30     10287
 46      9075
 10      9056
 38      7882
 2       6858
 11      6510
 50      6388
 56      5738
Name: count, dtype: int64

In [15]:
df["CHOSEN"].value_counts()

CHOSEN
0102500    4550
5310200    4083
5500700    4082
5500100    3958
1200500    3421
           ... 
2701503     580
5541001     575
2701403     566
2701402     523
4203207     509
Name: count, Length: 2351, dtype: int64

In [16]:
df["STAY"].value_counts()

STAY
True     3025964
False     188575
Name: count, dtype: int64

In [17]:
for c in df.columns:
    print(c)

YEAR
SAMPLE
SERIAL
CBSERIAL
NUMPREC
SUBSAMP
HHWT
HHTYPE
CLUSTER
ADJUST
CPI99
REGION
STATEICP
STATEFIP
COUNTYICP
COUNTYFIP
PUMA
CPUMA0010
CPUMA1020
DENSITY
METRO
PCTMETRO
MET2013
MET2013ERR
MET2023
MET2023ERR
METPOP10
METPOP20
CITY
CITYERR
CITYPOP
HOMELAND
STRATA
CNTRY
GQ
GQTYPE
GQTYPED
FARM
OWNERSHP
OWNERSHPD
MORTGAGE
MORTGAG2
FARMPROD
ACREHOUS
MORTAMT1
MORTAMT2
TAXINCL
INSINCL
PROPINSR
PROPTX99
OWNCOST
RENT
RENTGRS
RENTMEAL
CONDOFEE
MOBLHOME
COSTELEC
COSTGAS
COSTWATR
COSTFUEL
HHINCOME
FOODSTMP
VALUEH
SSMC
NFAMS
NSUBFAM
NCOUPLES
NMOTHERS
NFATHERS
MULTGEN
MULTGEND
CBNSUBFAM
RESPMODE
PERNUM
CBPERNUM
PERWT
SLWT
REPWTP
FAMUNIT
FAMSIZE
SUBFAM
SFTYPE
SFRELATE
CBSUBFAM
CBSFTYPE
CBSFRELATE
MOMLOC
MOMRULE
POPLOC
POPRULE
SPLOC
SPRULE
MOMLOC2
MOM2RULE
POPLOC2
POP2RULE
NCHILD
NCHLT5
NSIBS
ELDCH
YNGCH
RELATE
RELATED
SEX
AGE
BIRTHQTR
MARST
BIRTHYR
MARRNO
MARRINYR
YRMARR
DIVINYR
WIDINYR
FERTYR
RACE
RACED
HISPAN
HISPAND
BPL
BPLD
ANCESTR1
ANCESTR1D
ANCESTR2
ANCESTR2D
CITIZEN
YRNATUR
YRIMMIG
YRSUSA1
YRS

In [18]:
df["IS_CHILD_UNDER_6"] = np.where(df["AGE"] < 6, 1, 0)
df["IS_CHILD_6_TO_17"] = np.where((df["AGE"] >= 6) & (df["AGE"] <= 17), 1, 0)
df["IS_65_OR_OLDER"] = np.where(df["AGE"] >= 65, 1, 0)
df["IN_LF"] = np.where(df["EMPSTAT"].isin([1, 2]), 1, 0)
df["WORKING"] = np.where(df["EMPSTAT"] == 1, 1, 0)

In [ ]:
PRIMARY = {1, 2}
df["UNIT"] = np.where(
    df.SUBFAM != 0,
    # subfamily → its own unit
    df.CBSERIAL.astype(str) + "_SF" + df.SUBFAM.astype(str),
    np.where(
        # count primary, young children, unmarried partner
        df.RELATE.isin(PRIMARY)
        | (df.RELATE.isin([3]) & (df.AGE < 25))
        | df.RELATED.isin([1114]),
        # primary family, non-subfamily or child-category with age < 25
        df.CBSERIAL.astype(str) + "_P",
        # treat everyone else as singletons, individual decision units
        df.CBSERIAL.astype(str) + "_I" + df.PERNUM.astype(int).astype(str),
    ),
)

In [ ]:
df["_REF_PRIORITY"] = np.where((df["SFRELATE"] == 1) | (df["RELATE"] == 1), 0, 1)
df["_SEC_PRIORITY"] = np.where(
    (df["SFRELATE"] == 2) | (df["RELATE"] == 2) | (df["RELATED"] == 1114), 0, 1
)

units = df.groupby("UNIT").agg(
    REF_INDEX=("_REF_PRIORITY", "idxmin"),
    SEC_INDEX=("_SEC_PRIORITY", "idxmin"),
    NUM_CHILDREN_UNDER_6=("IS_CHILD_UNDER_6", "sum"),
    NUM_CHILDREN_6_TO_17=("IS_CHILD_6_TO_17", "sum"),
    SIZE=("_REF_PRIORITY", "size"),
    NUM_IN_IF=("IN_LF", "sum"),
    NUM_WORKING=("WORKING", "sum"),
)

In [ ]:
SHARED_COLS = [
    "GRADEATT",
    "RACE",
    "HISPAN",
    "BPL",
    "CITIZEN",
    "WORKING",
    "INDNAICS",
    "CLASSWKR",
    "AGE",
    "MARST",
    "DIVINYR",
    "WIDINYR",
    "MARRINYR",
    "VETSTATD",
    "EMPSTAT",
    "RACAMIND",
    "RACASIAN",
    "RACBLK",
    "RACPACIS",
    "RACWHT",
    "RACOTHER",
    "EDUC",
]
BASE_COLS = ["PERWT", "CHOSEN", "ORIGIN", "STAY"]
units[BASE_COLS] = df.loc[units["REF_INDEX"].values, BASE_COLS].values


units[[f"{c}_REF" for c in SHARED_COLS]] = df.loc[
    units["REF_INDEX"].values, SHARED_COLS
].values
units[[f"{c}_SEC" for c in SHARED_COLS]] = df.loc[
    units["REF_INDEX"].values, SHARED_COLS
].values

In [100]:
units_sample = units.sample(frac=sample_size, random_state=4703213)
units_sample.shape

(1750477, 55)

In [ ]:
units_sample["PAIRED_UNIT"] = np.where(
    units_sample["REF_INDEX"] != units_sample["SEC_INDEX"], 1, 0
)

In [ ]:
units_sample["CHILD_UNDER_6"] = np.where(units_sample["NUM_CHILDREN_UNDER_6"] > 0, 1, 0)
units_sample["CHILD_6_TO_17"] = np.where(units_sample["NUM_CHILDREN_6_TO_17"] > 0, 1, 0)
units_sample["CHILD"] = np.where(
    (units_sample["CHILD_UNDER_6"] == 1) | (units_sample["CHILD_6_TO_17"] == 1), 1, 0
)
units_sample["NUM_CHILDREN"] = (
    units_sample["NUM_CHILDREN_UNDER_6"] + units_sample["NUM_CHILDREN_6_TO_17"]
)

In [ ]:
units_sample["WORK2"] = np.where(
    units_sample["WORKING_REF"]
    & units_sample["WORKING_SEC"]
    & (units_sample["PAIRED_UNIT"] == 1),
    1,
    0,
)
units_sample["WORK1"] = np.where(
    (units_sample["WORKING_REF"] ^ units_sample["WORKING_SEC"])
    | (units_sample["PAIRED_UNIT"] == 0),
    1,
    0,
)
units_sample["PAIR_WORK1"] = np.where(
    units_sample["WORK1"] | (units_sample["PAIRED_UNIT"] == 1), 1, 0
)
units_sample["SINGLE_UNIT_WITH_CHILD"] = np.where(
    (units_sample["PAIRED_UNIT"] == 0) & (units_sample["CHILD"] == 1), 1, 0
)

In [ ]:
units_sample["MAX_EDUC"] = units_sample[["EDUC_REF", "EDUC_SEC"]].max(axis=1)
units_sample["EDU_NOHIGH"] = np.where(units_sample["MAX_EDUC"] <= 15, 1, 0)
units_sample["EDU_HIGH_BUT_NOT_BACHELORS"] = np.where(
    (units_sample["MAX_EDUC"] <= 20) & (units_sample["MAX_EDUC"] >= 16), 1, 0
)
units_sample["EDU_BACHELORS_OR_HIGHER"] = np.where(units_sample["MAX_EDUC"] >= 21, 1, 0)
units_sample["EDU_ONLY_HIGH"] = np.where(units_sample["MAX_EDUC"].isin([16, 17]), 1, 0)
units_sample["EDU_SOME_COLLEGE"] = np.where(
    units_sample["MAX_EDUC"].isin([18, 19, 20]), 1, 0
)
units_sample["EDU_ONLY_BACHELORS"] = np.where(units_sample["MAX_EDUC"] == 21, 1, 0)
units_sample["EDU_GRADUATE_DEG"] = np.where(units_sample["MAX_EDUC"] >= 22, 1, 0)
units_sample["EDU_HAS_DEGREE"] = np.where(units_sample["MAX_EDUC"] >= 21, 1, 0)
units_sample["EDU_NO_DEGREE"] = np.where(units_sample["MAX_EDUC"] <= 20, 1, 0)

In [ ]:
units_sample["MEAN_AGE"] = units_sample[["AGE_REF", "AGE_SEC"]].mean(axis=1)
units_sample["AGE_UNDER_18"] = np.where(units_sample["MEAN_AGE"] < 18, 1, 0)
units_sample["AGE_18_34"] = np.where(
    (units_sample["MEAN_AGE"] <= 34) & (units_sample["MEAN_AGE"] >= 18), 1, 0
)
units_sample["AGE_35_64"] = np.where(
    (units_sample["MEAN_AGE"] >= 35) & (units_sample["MEAN_AGE"] <= 64), 1, 0
)
units_sample["AGE_18_22"] = np.where(units_sample["MEAN_AGE"] <= 22, 1, 0)
units_sample["AGE_23_29"] = np.where(
    (units_sample["MEAN_AGE"] >= 23) & (units_sample["MEAN_AGE"] <= 29), 1, 0
)
units_sample["AGE_30_39"] = np.where(
    (units_sample["MEAN_AGE"] >= 30) & (units_sample["MEAN_AGE"] <= 39), 1, 0
)
units_sample["AGE_40_49"] = np.where(
    (units_sample["MEAN_AGE"] >= 40) & (units_sample["MEAN_AGE"] <= 49), 1, 0
)
units_sample["AGE_50_64"] = np.where(
    (units_sample["MEAN_AGE"] >= 50) & (units_sample["MEAN_AGE"] <= 64), 1, 0
)
units_sample["AGE_OVER_65"] = np.where((units_sample["MEAN_AGE"] >= 65), 1, 0)

In [ ]:
units_sample["FOREIGN_BORN"] = np.where(
    (units_sample["CITIZEN_REF"] >= 2) | (units_sample["BPL_SEC"] >= 2), 1, 0
)

In [ ]:
units_sample["IN_COLLEGE"] = np.where(
    (units_sample["GRADEATT_REF"] >= 6) | (units_sample["GRADEATT_SEC"] >= 6), 1, 0
)

In [ ]:
# this doesn't match the true MARST in df because some of the MARST people do not represent a decision unit
# this is true for most of these
units_sample["MARRIED"] = np.where(
    units_sample["MARST_REF"].isin([1, 2]) & units_sample["MARST_SEC"].isin([1, 2]),
    1,
    0,
)
# married unit and there are actually 2 people
units_sample["MARRIED_AND_TOGETHER"] = np.where(
    units_sample["MARRIED"] & (units_sample["PAIRED_UNIT"] == 1), 1, 0
)
# defined as either the reference or secondary person being divorced/widowed
units_sample["RECENTLY_WIDOWED_OR_DIVORCED"] = np.where(
    (units_sample["DIVINYR_REF"] == 2)
    | (units_sample["WIDINYR_REF"] == 2)
    | (units_sample["DIVINYR_SEC"] == 2)
    | (units_sample["DIVINYR_SEC"] == 2),
    1,
    0,
)
# referring to a recently married couple (or one of them, if they are living alone)
units_sample["RECENTLY_MARRIED"] = np.where(
    (units_sample["MARRINYR_REF"] == 2) & (units_sample["MARRINYR_SEC"] == 2), 1, 0
)

units_sample["MARRIED_MORE_THAN_YEAR"] = np.where(
    units_sample["MARRIED"] & ~units_sample["RECENTLY_MARRIED"], 1, 0
)

In [ ]:
units_sample["IN_MILITARY_REF"] = np.where(units_sample["VETSTATD_REF"] == 12, 1, 0)
units_sample["IN_MILITARY_SEC"] = np.where(units_sample["VETSTATD_SEC"] == 12, 1, 0)
units_sample["IN_MILITARY"] = np.where(
    (units_sample["IN_MILITARY_REF"] == 1) | (units_sample["IN_MILITARY_SEC"] == 11),
    1,
    0,
)
units_sample["UNEMPLOYED"] = np.where(
    (units_sample["EMPSTAT_REF"] == 2) & (units_sample["EMPSTAT_SEC"] == 2), 1, 0
)
units_sample["IN_LABOR_FORCE"] = np.where(
    units_sample["EMPSTAT_REF"].isin([1, 2]) | units_sample["EMPSTAT_SEC"].isin([1, 2]),
    1,
    0,
)
units_sample["NOT_IN_LABOR_FORCE"] = np.where(units_sample["IN_LABOR_FORCE"] == 1, 0, 1)

In [ ]:
RACE_GROUPS = {
    "WHITE": [1],
    "BLACK": [2],
    "INDIAN": [3],
    "AAPI": [4, 5, 6],
    "OTHER_RACE": [7, 8, 9],
}

for suffix in ["REF", "SEC"]:
    race = units_sample[f"RACE_{suffix}"]
    hisp = units_sample[f"HISPAN_{suffix}"]

    latino = hisp.ne(0) & hisp.ne(9)  # 9 = not reported, absent in 2018
    units_sample[f"LATINO_{suffix}"] = latino.astype(int)

    # Hispanic takes precedence, so the six categories stay mutually exclusive
    # and line up with the SE_B04001 area shares (non-Hispanic race + Hispanic).
    for name, codes in RACE_GROUPS.items():
        units_sample[f"{name}_{suffix}"] = (race.isin(codes) & ~latino).astype(int)

    units_sample[f"RACE_ETHNICITY_{suffix}"] = np.where(latino, 99, race)

/tmp/ipykernel_30605/3498442635.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  units_sample[f"RACE_ETHNICITY_{suffix}"] = np.where(latino, 99, race)
/tmp/ipykernel_30605/3498442635.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  units_sample[f"LATINO_{suffix}"] = latino.astype(int)
/tmp/ipykernel_30605/3498442635.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(a

In [ ]:
units_sample.to_parquet(
    f"pums_{sample_size * 100:.0f}_{year}.parquet", compression="gzip"
)

In [ ]:
# # Use SFRELATE if present, otherwise RELATE
# relate_col = "SFRELATE" if "SFRELATE" in df.columns else "RELATE"

# df["IS_SECONDARY"] = ((df["SFRELATE"] == 2) | (df["RELATE"] == 2)).astype(int)

# # Count per UNIT
# secondary_counts = (
#     df.groupby("UNIT", sort=False)["IS_SECONDARY"]
#     .sum()
#     .reset_index(name="N_SECONDARY")
# )
# print(secondary_counts["N_SECONDARY"].describe())
# print(((df["SFRELATE"] == 2) + (df["RELATE"] == 2)).max())

In [ ]:
# people who've recently had children category
# NOTE: this is a little iffy since this only applies to the women

# def add_recent_child_flag(df: pd.DataFrame) -> pd.DataFrame:
#     """FER_CL cleaning and inference for whether a household recently had a child."""
#     df["FER_CL"] = df["FER"].fillna(0)
#     df["FER_CL"] = np.where(df["FER_CL"] == 2, 0, df["FER_CL"])
#     rec_child = df.groupby("SERIALNO")["FER_CL"].max()
#     df["REC_CHILD"] = rec_child.loc[df["SERIALNO"]].values
#     return df
# df = lclean.add_recent_child_flag(df)
# df["FER_CL"].value_counts()